## Fase #4: Modelado / Modeling
En esta fase, se seleccionan y aplican diversas técnicas de modelado y se calibran sus parámetros a valores óptimos. Normalmente, hay varias técnicas para el mismo tipo de problema de minería de datos. Algunas técnicas tienen requisitos específicos en cuanto a la forma de los datos. Por lo tanto, a menudo es necesario volver a la fase de preparación dedatos.
* Seleccioná una variable objetivo que te permita cumplir el objetivo del estudio.
* Investigá un algoritmo de Ciencia de Datos que pueda ayudarte a predecir el comportamiento de esa variable. Explicá brevemente por qué te sería útil.
* Implementa el algoritmo.

### Variable objetivo y algoritmo

La variable objetivo es `agua_inyectada_m3`. Se elige regresión porque el volumen conserva más información y responde mejor a la necesidad logística que una etiqueta arbitraria de “alto/bajo”.

`RandomForestRegressor` combina muchos árboles y puede capturar relaciones no lineales e interacciones sin exigir una ecuación previa. Es adecuado para integrar variables numéricas y categóricas y representar distintos perfiles de operación. Se comparará contra `DummyRegressor(strategy="median")`: el modelo agrega valor si reduce el error de esa referencia sencilla. La prueba temporal 2025–2026 se mantiene intacta durante el entrenamiento.

In [ ]:
%pip install scikit-learn

In [ ]:
# @title Imports

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from comun import leer_dataframe_con_fechas


In [ ]:
# @title Preparación de variables

objetivo = "agua_inyectada_m3"
entradas_numericas = ["longitud_rama_horizontal_m", "cantidad_fracturas"]
entradas_categoricas = ["formacion_productiva", "subtipo_reservorio", "tipo_terminacion"]
entradas = entradas_numericas + entradas_categoricas


corte_temporal = pd.Timestamp("2025-01-01")
df_datos_energia = leer_dataframe_con_fechas()
datos_modelo = df_datos_energia[[*entradas, objetivo, "idpozo", "fecha_inicio_fractura", "id_base_fractura_adjiv"]].copy()
train = datos_modelo.loc[datos_modelo["fecha_inicio_fractura"] < corte_temporal].copy()
test = datos_modelo.loc[datos_modelo["fecha_inicio_fractura"] >= corte_temporal].copy()

pozos_compartidos = set(train["idpozo"]).intersection(test["idpozo"])
train = train.loc[~train["idpozo"].isin(pozos_compartidos)].copy()
if train.empty or test.empty:
	raise ValueError("La partición temporal quedó vacía; revisar cobertura de fechas.")

X_train, y_train = train[entradas], train[objetivo]
X_test, y_test = test[entradas], test[objetivo]


In [ ]:
codificador = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
prep = ColumnTransformer(
	[
		("num", SimpleImputer(strategy="median"), entradas_numericas),
		(
			"cat",
			Pipeline(
				[
					("imputer", SimpleImputer(strategy="most_frequent")),
					("onehot", codificador),
				]
			),
			entradas_categoricas,
		),
	]
)

modelo = Pipeline(
	[
		("prep", prep),
		("regresor", RandomForestRegressor(n_estimators=400, min_samples_leaf=3, random_state=42, n_jobs=-1)),
	]
)
referencia = DummyRegressor(strategy="median")

modelo.fit(X_train, y_train)
referencia.fit(X_train, y_train)
pred_modelo = modelo.predict(X_test)
pred_referencia = referencia.predict(X_test)

print(f"Entrenamiento: {len(train):,} registros hasta 2024")
print(f"Prueba temporal: {len(test):,} registros de 2025-2026")
print(f"Pozos retirados del entrenamiento por aparecer en prueba: {len(pozos_compartidos)}")


In [ ]:
from comun import exportar_modelos

exportar_modelos(
	modelo=modelo,
	referencia=referencia,
	X_test=X_test,
	y_test=y_test,
	test_metadata=test[
		[
			"id_base_fractura_adjiv",
			"idpozo",
			"fecha_inicio_fractura",
		]
	],
)

print("Modelos y datos de evaluación exportados.")


### Interpretación de la partición

El entrenamiento utiliza **1.999 operaciones anteriores a 2025** y la prueba conserva **623 operaciones de 2025–2026**. Se retiraron del entrenamiento cuatro pozos que reaparecen en prueba. Esta separación es más exigente y más cercana a un uso futuro que una división aleatoria, porque el modelo aprende del pasado y se evalúa en un período posterior. También permite detectar cambios de tecnología o de prácticas operativas que una partición aleatoria podría ocultar.